# Convert CSV to Parquet

In [33]:
import pandas as pd

## Download weather data

First, we need to download the data.

sample download link for Sydney Airport:
https://www.bom.gov.au/climate/dwo/202605/text/IDCJDW2125.202605.csv
https://www.bom.gov.au/climate/dwo/202605/text/IDCJDW2119.202605.csv

In [34]:
base_download_link = 'https://www.bom.gov.au/climate/dwo/'
dates = ['202605', '202604', '202603', '202602', '202601',
        '202512', '202511', '202510', '202509', '202508',
        '202507', '202506', '202505', '202504', '202503',
        '202502', '202501']
locations = ['IDCJDW2125', 'IDCJDW2119', #KZ
            'IDCJDW2012', 'IDCJDW2024',  #DH
            'IDCJDW2101', 'IDCJDW2008',  #JY
            'IDCJDW2139', 'IDCJDW2027',  #AA
            'IDCJDW2014', 'IDCJDW2087'] #OM

def get_download_link(location, date):
    """
    download the data for a specific location and date.
    """
    return base_download_link + date + '/text/' + location + '.' + date + '.csv'

def download(out_dir):
    """
    Download weather data CSV files for all locations and dates.

    Args:
        out_dir: Output directory to save the downloaded files
    """
    import os
    import requests
    import time

    os.makedirs(out_dir, exist_ok=True)

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'max-age=0'
    }

    for location in locations:
        for date in dates:
            url = get_download_link(location, date)
            filename = f"{location}.{date}.csv"
            filepath = os.path.join(out_dir, filename)

            try:
                response = requests.get(url, headers=headers, timeout=30)
                if response.status_code == 200:
                    with open(filepath, 'wb') as f:
                        f.write(response.content)
                    print(f"Downloaded: {filename}")
                else:
                    print(f"Failed to download {filename}: HTTP {response.status_code}")
            except Exception as e:
                print(f"Error downloading {filename}: {e}")

            time.sleep(0.5)

    print("Download complete!")


In [35]:
download("./weather_data")

Downloaded: IDCJDW2125.202605.csv
Downloaded: IDCJDW2125.202604.csv
Downloaded: IDCJDW2125.202603.csv
Downloaded: IDCJDW2125.202602.csv
Downloaded: IDCJDW2125.202601.csv
Downloaded: IDCJDW2125.202512.csv
Downloaded: IDCJDW2125.202511.csv
Downloaded: IDCJDW2125.202510.csv
Downloaded: IDCJDW2125.202509.csv
Downloaded: IDCJDW2125.202508.csv
Downloaded: IDCJDW2125.202507.csv
Downloaded: IDCJDW2125.202506.csv
Downloaded: IDCJDW2125.202505.csv
Downloaded: IDCJDW2125.202504.csv
Downloaded: IDCJDW2125.202503.csv
Failed to download IDCJDW2125.202502.csv: HTTP 404
Failed to download IDCJDW2125.202501.csv: HTTP 404
Downloaded: IDCJDW2119.202605.csv
Downloaded: IDCJDW2119.202604.csv
Downloaded: IDCJDW2119.202603.csv
Downloaded: IDCJDW2119.202602.csv
Downloaded: IDCJDW2119.202601.csv
Downloaded: IDCJDW2119.202512.csv
Downloaded: IDCJDW2119.202511.csv
Downloaded: IDCJDW2119.202510.csv
Downloaded: IDCJDW2119.202509.csv
Downloaded: IDCJDW2119.202508.csv
Downloaded: IDCJDW2119.202507.csv
Downloaded: ID

## Combine data to parquet file

Now, we can read the csv files and combine them into a single parquet file.

In [36]:
def combine_data(csv_dir="./weather_data", output_file="./weather_data.parquet"):
    """
    Read all CSV files from the directory and combine them into a single parquet file.
    
    Args:
        csv_dir: Directory containing the CSV files
        output_file: Path to save the combined parquet file
    """
    import os
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    
    csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
    
    if not csv_files:
        print(f"No CSV files found in {csv_dir}")
        return
    
    def find_header_row(filepath):
        """Find the row number where the actual data header starts."""
        with open(filepath, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                if line.startswith(',"Date"'):
                    return i
        return None
    
    dataframes = []
    for csv_file in csv_files:
        filepath = os.path.join(csv_dir, csv_file)
        try:
            header_row = find_header_row(filepath)
            if header_row is None:
                print(f"Could not find header in {csv_file}")
                continue
            df = pd.read_csv(filepath, encoding='latin-1', skiprows=header_row, low_memory=False)
            df['source_file'] = csv_file
            dataframes.append(df)
            print(f"Loaded: {csv_file}")
        except Exception as e:
            print(f"Error loading {csv_file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        
        for col in combined_df.columns:
            if combined_df[col].dtype == 'object':
                combined_df[col] = combined_df[col].astype(str)
        
        table = pa.Table.from_pandas(combined_df)
        pq.write_table(table, output_file)
        
        print(f"Combined {len(dataframes)} files into {output_file}")
        print(f"Total rows: {len(combined_df)}")
        return combined_df
    else:
        print("No data to save")
        return None

In [37]:
combine_data()

Loaded: IDCJDW2012.202602.csv
Loaded: IDCJDW2139.202604.csv
Loaded: IDCJDW2125.202605.csv
Loaded: IDCJDW2087.202604.csv
Loaded: IDCJDW2101.202507.csv
Loaded: IDCJDW2101.202506.csv
Loaded: IDCJDW2101.202512.csv
Loaded: IDCJDW2087.202605.csv
Loaded: IDCJDW2125.202604.csv
Loaded: IDCJDW2027.202503.csv
Loaded: IDCJDW2139.202605.csv
Loaded: IDCJDW2012.202603.csv
Loaded: IDCJDW2012.202601.csv
Loaded: IDCJDW2101.202504.csv
Loaded: IDCJDW2101.202510.csv
Loaded: IDCJDW2101.202511.csv
Loaded: IDCJDW2101.202505.csv
Loaded: IDCJDW2012.202604.csv
Loaded: IDCJDW2139.202602.csv
Loaded: IDCJDW2027.202504.csv
Loaded: IDCJDW2027.202510.csv
Loaded: IDCJDW2125.202603.csv
Loaded: IDCJDW2087.202602.csv
Loaded: IDCJDW2087.202603.csv
Loaded: IDCJDW2125.202602.csv
Loaded: IDCJDW2027.202511.csv
Loaded: IDCJDW2027.202505.csv
Loaded: IDCJDW2139.202603.csv
Loaded: IDCJDW2012.202605.csv
Loaded: IDCJDW2024.202509.csv
Loaded: IDCJDW2139.202601.csv
Loaded: IDCJDW2027.202507.csv
Loaded: IDCJDW2087.202601.csv
Loaded: ID

,Unnamed: 0,Date,Minimum temperature (°C),Maximum temperature (°C),Rainfall (mm),Evaporation (mm),Sunshine (hours),Direction of maximum wind gust,Speed of maximum wind gust (km/h),Time of maximum wind gust,...,9am wind direction,9am wind speed (km/h),9am MSL pressure (hPa),3pm Temperature (°C),3pm relative humidity (%),3pm cloud amount (oktas),3pm wind direction,3pm wind speed (km/h),3pm MSL pressure (hPa),source_file
0,NaN,2026-02-1,NaN,NaN,NaN,NaN,NaN,SSW,65.0,14:23,...,W,13,1007.1,NaN,NaN,NaN,SW,33,1004.4,IDCJDW2012.202602.csv
1,NaN,2026-02-2,NaN,NaN,NaN,NaN,NaN,E,46.0,16:38,...,SE,26,1015.4,NaN,NaN,NaN,SE,22,1017.4,IDCJDW2012.202602.csv
2,NaN,2026-02-3,NaN,NaN,NaN,NaN,NaN,ENE,39.0,01:53,...,E,24,1022.5,NaN,NaN,NaN,N,20,1020.1,IDCJDW2012.202602.csv
3,NaN,2026-02-4,NaN,NaN,NaN,NaN,NaN,W,46.0,14:19,...,W,2,1019.6,NaN,NaN,NaN,W,22,1016.0,IDCJDW2012.202602.csv
4,NaN,2026-02-5,NaN,NaN,NaN,NaN,NaN,N,43.0,14:41,...,NNW,9,1018.3,NaN,NaN,NaN,N,26,1015.4,IDCJDW2012.202602.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4365,NaN,2026-05-7,9.0,18.5,0.0,NaN,NaN,W,57.0,11:21,...,N,9,1014.0,16.1,30.0,NaN,WSW,30.0,1015.5,IDCJDW2008.202605.csv
4366,NaN,2026-05-8,8.0,21.0,0.0,NaN,NaN,W,46.0,11:28,...,N,9,1018.4,20.6,37.0,NaN,W,20.0,1016.1,IDCJDW2008.202605.csv
4367,NaN,2026-05-9,9.3,22.5,0.0,NaN,NaN,ESE,30.0,14:34,...,W,11,1025.4,20.3,49.0,NaN,ESE,20.0,1023.1,IDCJDW2008.202605.csv
4368,NaN,2026-05-10,8.5,23.5,0.0,NaN,NaN,SE,31.0,15:54,...,WNW,11,1029.5,21.5,56.0,NaN,SE,20.0,1027.8,IDCJDW2008.202605.csv
